# Bull Flag Continuation Backtest

这份 notebook 直接读取 `Dataframes/stock_price.csv`，用于回测和研究标准 bull flag continuation 策略。

默认流程：

1. 读取并清洗价格数据
2. 生成 `BullFlagContinuationResearcher.trade_df`
3. 把 `trade_df` 交给 `TradePlanBacktester` 跑回测
4. 查看 summary、单笔样本和参数扫描结果


In [ ]:
# Notebook bootstrap: repo root + sys.path
import sys
from pathlib import Path

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "strategies").exists() and (path / "backtester").exists() and (path / "Dataframes").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate repo root from the current notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from backtester import TradePlanBacktester
from score_system.bull_flag_grid_search import run_bull_flag_grid_search
from strategies.bull_flag_continuation import BullFlagContinuationResearcher, BullFlagStrategyConfig

PRICE_PATH = REPO_ROOT / 'Dataframes' / 'stock_price.csv'
PRICE_PATH

In [ ]:
stock_candle_df = pd.read_csv(PRICE_PATH)
if 'Unnamed: 0' in stock_candle_df.columns:
    stock_candle_df = stock_candle_df.drop(columns=['Unnamed: 0'])
stock_candle_df['date'] = pd.to_datetime(stock_candle_df['date'])
if 'constituent_trade_date' in stock_candle_df.columns:
    stock_candle_df['constituent_trade_date'] = pd.to_datetime(stock_candle_df['constituent_trade_date'])
stock_candle_df = stock_candle_df.sort_values(['date', 'ticker'], kind='mergesort').reset_index(drop=True)
stock_candle_df.head()

In [ ]:
config = BullFlagStrategyConfig()
researcher = BullFlagContinuationResearcher(stock_candle_df, config=config)
trade_df = researcher.trade_df.copy()

pd.Series({
    'rows': len(researcher.stock_candle_df),
    'signal_candles': int(researcher.stock_candle_df['signal_candle'].sum()),
    'entry_signals': int(researcher.stock_candle_df['entry_signal'].sum()),
    'planned_trades': len(trade_df),
})

## 回测

这里优先复用 `researcher.trade_df`，避免 backtester 再次重算 researcher。

In [ ]:
backtester = TradePlanBacktester(
    stock_candle_df,
    trade_df=trade_df,
    initial_capital=1_000_000.0,
    fixed_entry_notional=20_000.0,
    board_lot_size=100,
)

results = backtester.compute_metrics(start_date='2020-01-01', end_date='2026-03-16')
display(pd.Series(results['summary']))
display(pd.Series(results.get('trade_summary', {})))
results['figure']

## 单笔样本

抽一笔样本交易，查看信号、上下文和图形。

In [ ]:
if trade_df.empty:
    sample_signal = None
else:
    sample_signal = trade_df.iloc[0]

sample_signal

In [ ]:
if sample_signal is not None:
    inspection = researcher.inspect_signal(sample_signal['ticker'], sample_signal['signal_date'])
    display(pd.Series(inspection['summary']))
    display(inspection['signal_row'])
    researcher.plot_signal_context(sample_signal['ticker'], sample_signal['signal_date'])
else:
    print('No trade found in trade_df.')

## 参数扫描

这里用一个小网格快速观察这两个参数对结果的影响：

- `min_reward_r`
- `measured_move_fraction`


In [ ]:
grid_results = run_bull_flag_grid_search(
    stock_candle_df,
    param_grid={
        'min_reward_r': [1.0, 1.5, 2.0],
        'measured_move_fraction': [0.5, 0.75, 1.0],
    },
    base_config=config,
    start_date='2020-01-01',
    end_date='2026-03-16',
    initial_capital=1_000_000.0,
    fixed_entry_notional=20_000.0,
    board_lot_size=100,
)

display(grid_results['summary'])

In [ ]:
grid_results['figure']